In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
import pandas as pd
import numpy as np

In [9]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [12]:
path = '/workspaces/crmprtd/update_sheet/sheet_24Dec/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges (2).xlsx', header = 1)   # pandas automatically uses openpyxl

# Select rows where ISSUE contains "Attribution" (case-sensitive)
df_del_ref = df[
    df["ISSUE"].str.contains("Delete Reference", na=False) 
]

df_del_ref =  df_del_ref[['ISSUE', 'Station ID', 'Unique Names', 'Native ID']].reset_index(drop=True)


df_del_ref["Station ID"] = pd.to_numeric(df_del_ref["Station ID"], errors="coerce").astype("Int64")

split_ids = df_del_ref['Native ID'].astype(str).str.split('->', expand=True)

df_del_ref['old_native_id'] = split_ids[0].str.strip()
df_del_ref['new_native_id'] = split_ids[1].str.strip()

df_del_ref = df_del_ref.drop(columns=['Native ID'])
df_del_ref["Station ID"] = pd.to_numeric(df_del_ref["Station ID"], errors="coerce").astype("Int64")

df_del_ref["ref_station_id"] = df_del_ref["ISSUE"].str.extract(r'(\d+)$').astype(int)

df_del_ref = df_del_ref.rename(columns={'Station ID': 'old_station_id'})


len(df_del_ref)

df_del_ref

,ISSUE,old_station_id,Unique Names,old_native_id,new_native_id,ref_station_id
0,"Attribution, Delete Reference to 12532",12090,Burnaby Mountain,E206270,T014,12532
1,"Attribution, Delete Reference to 12536",2644,Chilliwack Airport,E220891,T012,12536
2,"Attribution, Delete Reference to 12549",1588,Hope Airport,E223756,T029,12549
3,"Attribution, Delete Reference to 12556",12191,Langley Central,E209178,T027,12556
4,"Attribution, Delete Reference to 12558",12200,Mission School Works Yard,E302130,T043,12558
5,"Attribution, Delete Reference to 12600",12196,Maple Ridge Golden Ears School,E232245,T028,12600
6,"Attribution, Delete Reference to 12601",12204,North Burnaby Capitol Hill -> Burnaby-Capitol ...,E244516,T023,12601
7,"Attribution, Delete Reference to 12603",12207,North Vancouver Mahon Park -> N. Vancouver-Mah...,E209177,T026,12603
8,"Attribution, Delete Reference to 2634",12160,Horseshoe Bay,E275843,T035,2634
9,"Attribution,Delete Reference to 12593",12091,Burnaby North Eton,E244517,T024,12593


In [17]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE m.station_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE m.station_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE m_old.station_id = %s
       AND m_new.station_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

def count_station_stats(
    old_station_id,
    new_station_id,
    engine
):
    params = (
        # n_old
        old_station_id,

        # n_new
        new_station_id,

        # n_overlap
        old_station_id,
        new_station_id,

        # n_overlap_same_datum
        old_station_id,
        new_station_id,
    )

    df = pd.read_sql(query, engine, params=params)
    return df.iloc[0]

stats = df_del_ref.apply(
    lambda r: count_station_stats(
        r["old_station_id"],
        r["ref_station_id"],
        engine
    ),
    axis=1
)

df_del_ref[[
    "n_old",
    "n_new",
    "n_overlap",
    "n_overlap_same_datum"
]] = stats

df_del_ref

,ISSUE,old_station_id,Unique Names,old_native_id,new_native_id,ref_station_id,n_old,n_new,n_overlap,n_overlap_same_datum
0,"Attribution, Delete Reference to 12532",12090,Burnaby Mountain,E206270,T014,12532,0,5620,0,0
1,"Attribution, Delete Reference to 12536",2644,Chilliwack Airport,E220891,T012,12536,0,160146,0,0
2,"Attribution, Delete Reference to 12549",1588,Hope Airport,E223756,T029,12549,0,5558,0,0
3,"Attribution, Delete Reference to 12556",12191,Langley Central,E209178,T027,12556,0,5622,0,0
4,"Attribution, Delete Reference to 12558",12200,Mission School Works Yard,E302130,T043,12558,0,5654,0,0
5,"Attribution, Delete Reference to 12600",12196,Maple Ridge Golden Ears School,E232245,T028,12600,0,2822,0,0
6,"Attribution, Delete Reference to 12601",12204,North Burnaby Capitol Hill -> Burnaby-Capitol ...,E244516,T023,12601,0,2825,0,0
7,"Attribution, Delete Reference to 12603",12207,North Vancouver Mahon Park -> N. Vancouver-Mah...,E209177,T026,12603,0,2827,0,0
8,"Attribution, Delete Reference to 2634",12160,Horseshoe Bay,E275843,T035,2634,0,28055,0,0
9,"Attribution,Delete Reference to 12593",12091,Burnaby North Eton,E244517,T024,12593,0,2338,0,0


In [21]:
query = """
SELECT
    -- old count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s
    ) AS n_old,

    -- new count
    (SELECT COUNT(*)
     FROM meta_history m
     JOIN meta_station s ON m.station_id = s.station_id
     JOIN obs_raw o ON m.history_id = o.history_id
     WHERE s.native_id = %s
    ) AS n_new,

    -- overlap count (time + variable)
    (SELECT COUNT(*)
     FROM (
         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s

         INTERSECT

         SELECT o.obs_time, o.vars_id
         FROM meta_history m
         JOIN meta_station s ON m.station_id = s.station_id
         JOIN obs_raw o ON m.history_id = o.history_id
         WHERE s.native_id = %s
     ) t
    ) AS n_overlap,

    -- overlap with identical datum
    (SELECT COUNT(*)
     FROM meta_history m_old
     JOIN meta_station s_old ON m_old.station_id = s_old.station_id
     JOIN obs_raw o_old ON m_old.history_id = o_old.history_id

     JOIN meta_history m_new
     JOIN meta_station s_new ON m_new.station_id = s_new.station_id
     JOIN obs_raw o_new ON m_new.history_id = o_new.history_id

       ON o_old.obs_time = o_new.obs_time
      AND o_old.vars_id  = o_new.vars_id

     WHERE s_old.native_id = %s
       AND s_new.native_id = %s
       AND ABS(o_old.datum - o_new.datum) < 0.01
    ) AS n_overlap_same_datum;
"""

stats = df_del_ref.apply(
    lambda r: count_station_stats(
        r["old_native_id"],
        r["new_native_id"],  # your new station's native_id
        engine
    ),
    axis=1
)

df_del_ref[[
    "n_old",
    "n_new",
    "n_overlap",
    "n_overlap_same_datum"
]] = stats

df_del_ref

,ISSUE,old_station_id,Unique Names,old_native_id,new_native_id,ref_station_id,n_old,n_new,n_overlap,n_overlap_same_datum
0,"Attribution, Delete Reference to 12532",12090,Burnaby Mountain,E206270,T014,12532,0,5620,0,0
1,"Attribution, Delete Reference to 12536",2644,Chilliwack Airport,E220891,T012,12536,0,160146,0,0
2,"Attribution, Delete Reference to 12549",1588,Hope Airport,E223756,T029,12549,0,5558,0,0
3,"Attribution, Delete Reference to 12556",12191,Langley Central,E209178,T027,12556,0,5622,0,0
4,"Attribution, Delete Reference to 12558",12200,Mission School Works Yard,E302130,T043,12558,0,5654,0,0
5,"Attribution, Delete Reference to 12600",12196,Maple Ridge Golden Ears School,E232245,T028,12600,0,2822,0,0
6,"Attribution, Delete Reference to 12601",12204,North Burnaby Capitol Hill -> Burnaby-Capitol ...,E244516,T023,12601,0,2825,0,0
7,"Attribution, Delete Reference to 12603",12207,North Vancouver Mahon Park -> N. Vancouver-Mah...,E209177,T026,12603,0,2827,0,0
8,"Attribution, Delete Reference to 2634",12160,Horseshoe Bay,E275843,T035,2634,0,28055,0,0
9,"Attribution,Delete Reference to 12593",12091,Burnaby North Eton,E244517,T024,12593,0,2338,0,0


In [22]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.native_id = :native_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in df_del_ref.iterrows():
        native_id = row['old_native_id']

        exists = conn.execute(
            exists_sql,
            {"native_id": native_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(df_del_ref)}] "
            f"old_native_id={native_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
df_del_ref['old_station_status'] = exists_list

[  1/12] old_native_id=E206270         | → EXISTS
[  2/12] old_native_id=E220891         | → EXISTS
[  3/12] old_native_id=E223756         | → EXISTS
[  4/12] old_native_id=E209178         | → EXISTS
[  5/12] old_native_id=E302130         | → EXISTS
[  6/12] old_native_id=E232245         | → EXISTS
[  7/12] old_native_id=E244516         | → EXISTS
[  8/12] old_native_id=E209177         | → EXISTS
[  9/12] old_native_id=E275843         | → EXISTS
[ 10/12] old_native_id=E244517         | → EXISTS
[ 11/12] old_native_id=E244515         | → EXISTS
[ 12/12] old_native_id=Quesnel Senior Secondary | → NOT EXISTS


In [23]:
from sqlalchemy import text

exists_sql = text("""
SELECT EXISTS (
    SELECT 1
    FROM meta_history m
    JOIN meta_station s ON m.station_id = s.station_id
    WHERE s.station_id = :station_id
)
""")

exists_list = []

with engine.connect() as conn:   # reuse ONE connection
    for i, row in df_del_ref.iterrows():
        station_id = row['old_station_id']

        exists = conn.execute(
            exists_sql,
            {"station_id": station_id}
        ).scalar()

        status = "EXISTS" if exists else "NOT EXISTS"
        exists_list.append(status)

        print(
            f"[{i+1:>3}/{len(df_del_ref)}] "
            f"old_station_id={station_id:<15} | "
            f"→ {status}"
        )

# add column to DataFrame
df_del_ref['old_station_status'] = exists_list

[  1/12] old_station_id=12090           | → EXISTS
[  2/12] old_station_id=2644            | → EXISTS
[  3/12] old_station_id=1588            | → EXISTS
[  4/12] old_station_id=12191           | → EXISTS
[  5/12] old_station_id=12200           | → EXISTS
[  6/12] old_station_id=12196           | → EXISTS
[  7/12] old_station_id=12204           | → EXISTS
[  8/12] old_station_id=12207           | → EXISTS
[  9/12] old_station_id=12160           | → EXISTS
[ 10/12] old_station_id=12091           | → EXISTS
[ 11/12] old_station_id=12086           | → EXISTS
[ 12/12] old_station_id=12943           | → NOT EXISTS
